# Visium Cortical Structure Analysis

**Biological Question:** Can unsupervised gene expression clustering recover anatomically distinct tissue regions in mouse brain Visium data without cell type deconvolution?

**Dataset:** 10x Genomics Visium mouse brain section, loaded via Squidpy

**Tools:** Scanpy, Squidpy

**Key limitation to keep in mind:** Each Visium spot captures RNA from multiple cells simultaneously. Clusters reflect mixtures of cell types, not pure populations.

## Step 1: Import libraries and load dataset

In [ ]:
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import os

# Set figures folder
os.makedirs('../figures', exist_ok=True)
sc.settings.figdir = '../figures/'
sc.settings.verbosity = 1

# Load Visium mouse brain dataset directly from Squidpy
# No manual download needed
adata = sq.datasets.visium_hne_adata()

print(adata)
print(f'Spots: {adata.n_obs}')
print(f'Genes: {adata.n_vars}')

## Step 2: Quality Control

We remove low quality spots before any analysis.

- Spots with fewer than 200 detected genes are likely empty or damaged
- Genes detected in fewer than 10 spots carry no useful signal
- High mitochondrial content indicates damaged tissue, not a biological signal

In [ ]:
# Identify mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('mt-')

# Calculate QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

# Visualize QC metrics before filtering
sc.pl.violin(
    adata,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    jitter=0.4,
    save='_qc_metrics.png'
)

# Apply filters
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=10)
adata = adata[adata.obs['pct_counts_mt'] < 20]

print(f'Spots after QC: {adata.n_obs}')
print(f'Genes after QC: {adata.n_vars}')

## Step 3: Normalization

Library size normalization corrects for technical differences in total RNA captured per spot.
Log transformation compresses the dynamic range so highly expressed genes do not dominate clustering.

In [ ]:
# Save raw counts before normalization
adata.raw = adata

# Normalize to 10,000 counts per spot
sc.pp.normalize_total(adata, target_sum=1e4)

# Log transform
sc.pp.log1p(adata)

print('Normalization complete')

## Step 4: Highly Variable Gene Selection

Retaining the top 2,000 most variable genes removes noise from lowly expressed genes
and keeps memory usage within 4 GB RAM.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
adata = adata[:, adata.var['highly_variable']]
print(f'Highly variable genes selected: {adata.n_vars}')

## Step 5: PCA

Compresses 2,000-gene space into 50 principal components.
Captures most variance while making downstream steps faster and less noisy.

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50)
sc.pl.pca_variance_ratio(adata, n_pcs=50, save='_pca_variance.png')
print('PCA complete')

## Step 6: Clustering

Leiden clustering at resolution 0.5 balances biological granularity
against over-fragmentation of small tissue regions.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

sc.pl.umap(adata, color='leiden', save='_umap_clusters.png')

print(f'Number of clusters: {adata.obs["leiden"].nunique()}')

## Step 7: Spatial Visualization

This is the primary test of the project question.
Do expression-based clusters align with anatomical regions in the H&E image?

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color='leiden',
    save='../figures/spatial_clusters.png'
)
print('Spatial cluster plot saved')

## Step 8: Spatially Variable Genes

Moran I measures spatial autocorrelation for each gene.
High Moran I means expression is spatially organized, not randomly distributed.
These genes reveal which molecular programs are tied to specific tissue regions.

In [ ]:
sq.gr.spatial_neighbors(adata)
sq.gr.spatial_autocorr(adata, mode='moran', n_perms=100, n_jobs=1)

top_svgs = adata.uns['moranI'].head(10)
print('Top spatially variable genes:')
print(top_svgs)

top_genes = top_svgs.index[:4].tolist()
sq.pl.spatial_scatter(
    adata,
    color=top_genes,
    save='../figures/spatial_gene_map.png'
)
print('Spatial gene map saved')

## Critical Interpretation

If clusters align with anatomical regions, this supports the hypothesis that
gene expression alone recovers spatial tissue organization.

However two alternative explanations must always be considered:

1. Technical capture gradients: Clusters may reflect differences in RNA capture
efficiency due to tissue thickness or preservation quality, not true biological differences.

2. Spot mixing: Each spot contains multiple cell types. Cluster identity reflects
the dominant cell type mixture, not a pure population.
Deconvolution with Cell2location is the required next step to confirm cell type identity.